In [1]:
import os
import sys
import argparse
import time
import threading
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from torchvision.utils import save_image
try:
    import psutil
except ImportError:
    psutil = None
try:
    import pynvml
    pynvml.nvmlInit()
    GPU_MON_AVAIL = True
except:
    GPU_MON_AVAIL = False

#### Для запуска и обучения моделей вроде GAN, VAE и DPM в условиях генерации мультимедийных данных нужно задать гиперпараметры для выделения ресурсов.

In [2]:
DATA_ROOT = "captcha"                 # папка, содержащая train/ и test/
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
WEIGHTS_DIR = "weights"
GEN_DIRS = {
    "gan": "gan_gen_captcha",
    "vae": "vae_gen_captcha",
    "ddpm": "ddpm_gen_captcha"
}
IMG_SIZE = (200, 50)#W, H
LATENT_DIM = 100
BATCH_SIZE = 32
NUM_EPOCHS_GAN = 50
NUM_EPOCHS_VAE = 50
NUM_EPOCHS_DDPM = 30  #диффузия медленнее
NUM_EPOCHS_CRNN = 30
LEARNING_RATE = 2e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHARS = "abcdefghijklmnopqrstuvwxyz0123456789" #символы, встречающиеся в каптче
CHAR_TO_IDX = {c: i for i, c in enumerate(CHARS)}
IDX_TO_CHAR = {i: c for c, i in CHAR_TO_IDX.items()}
N_CLASSES = len(CHARS) + 1 #+1 для blank CTC
CAPTCHA_LEN = 6 #все каптчи из 6 символов

#### Необходимо предобработать изображения, так как в отличие от табличных данных они не являются единой таблицей, а предтавляют собой множество разрозненных изображенй, которые могут иметь разные расширения, метки и метаданные.

In [3]:
class CaptchaDataset(Dataset):
    def __init__(self, root_dir, return_label=True):
        self.return_label = return_label
        self.image_paths = []
        self.labels = []
        for fname in sorted(os.listdir(root_dir)):
            if fname.lower().endswith(".jpeg") or fname.lower().endswith(".jpg"):
                path = os.path.join(root_dir, fname)
                stem = os.path.splitext(fname)[0]
                parts = stem.split("_", 1)
                if len(parts) != 2:
                    continue
                label = parts[1].lower()
                if len(label) != CAPTCHA_LEN:
                    continue
                if not all(c in CHAR_TO_IDX for c in label):
                    continue
                self.image_paths.append(path)
                self.labels.append(label)
        self.transform = T.Compose([
            T.Resize((IMG_SIZE[1], IMG_SIZE[0])), #H, W
            T.ToTensor(), #[0,1]
            T.Normalize((0.5,), (0.5,)) #-> [-1,1] для генеративных
        ])
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.transform(img)
        if self.return_label:
            label = self.labels[idx]
            label_indices = [CHAR_TO_IDX[c] for c in label]
            return img, torch.tensor(label_indices, dtype=torch.long)
        return img

#### Так как модели долго учатся, целесообразно сохранять их веса после определённого количества эпох, чтобы потом их можно было подгрузить и продолжить выполнять обучение на других данных (это и есть retraining).

In [4]:
def create_dir(path):
    os.makedirs(path, exist_ok=True)
def save_loss_plot(losses, title, filename):
    plt.figure()
    plt.plot(losses)
    plt.title(title)
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.savefig(filename)
    plt.close()
def weights_exist(model_name):
    path = os.path.join(WEIGHTS_DIR, f"{model_name}_weights.pth")
    return os.path.isfile(path)
def save_weights(model, model_name):
    create_dir(WEIGHTS_DIR)
    path = os.path.join(WEIGHTS_DIR, f"{model_name}_weights.pth")
    torch.save(model.state_dict(), path)
    print(f"Веса {model_name} сохранены в {path}")
def load_weights(model, model_name):
    path = os.path.join(WEIGHTS_DIR, f"{model_name}_weights.pth")
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.to(DEVICE)
    print(f"Веса {model_name} загружены из {path}")
    return model

#### Для наблюдения за процессом обучения можно создать мониторы, отслеживающие используемые ресурсы компьютера и метрики качетва модели на каждой эпохе обучения.

In [5]:
class ResourceMonitor(threading.Thread):
    def __init__(self, interval=1.0):
        super().__init__()
        self.interval = interval
        self.running = False
        self.cpu_usage = []
        self.gpu_usage = []
        self.gpu_mem = []
        self.ram_usage = []
    def run(self):
        self.running = True
        while self.running:
            if psutil:
                self.cpu_usage.append(psutil.cpu_percent())
                self.ram_usage.append(psutil.virtual_memory().percent)
            if GPU_MON_AVAIL:
                handle = pynvml.nvmlDeviceGetHandleByIndex(0)
                util = pynvml.nvmlDeviceGetUtilizationRates(handle)
                mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
                self.gpu_usage.append(util.gpu)
                self.gpu_mem.append(mem.used / 1024**2)   # МБ
            time.sleep(self.interval)
    def stop(self):
        self.running = False
    def summary(self):
        s = ""
        if self.cpu_usage:
            s += f"CPU usage max: {max(self.cpu_usage):.1f}%, avg: {np.mean(self.cpu_usage):.1f}%\n"
        if self.ram_usage:
            s += f"RAM usage max: {max(self.ram_usage):.1f}%, avg: {np.mean(self.ram_usage):.1f}%\n"
        if self.gpu_usage:
            s += f"GPU util max: {max(self.gpu_usage):.1f}%, avg: {np.mean(self.gpu_usage):.1f}%\n"
        if self.gpu_mem:
            s += f"GPU mem max: {max(self.gpu_mem):.0f} MB, avg: {np.mean(self.gpu_mem):.0f} MB\n"
        return s

#### Для реализации обучения и мониторинга необходимо задать метрики качества.

In [6]:
class InceptionFeatureExtractor(nn.Module):
    def __init__(self): #признаки из inception_v3 (2048-d)
        super().__init__()
        inception = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT, transform_input=False)
        inception.fc = nn.Identity()
        self.model = inception.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.resize = T.Compose([
            T.Resize((299, 299)),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    def forward(self, x):
        x = (x + 1) / 2 #x in [-1,1] -> [0,1] -> нормировка для inception
        x = self.resize(x)
        return self.model(x)
def calculate_fid(real_loader, fake_images, feat_extractor, device): #Упрощённый FID: сравнивает статистики признаков реальных и сгенерированных изображений
    feat_extractor.to(device)
    real_feats = []
    with torch.no_grad():
        for imgs, _ in real_loader:
            imgs = imgs.to(device)
            feats = feat_extractor(imgs).cpu().numpy()
            real_feats.append(feats)
    real_feats = np.concatenate(real_feats, axis=0)
    mu_real = np.mean(real_feats, axis=0)
    sigma_real = np.cov(real_feats, rowvar=False)
    fake_feats = []
    for img in fake_images: #fake_images: список тензоров [C,H,W]
        img_batch = img.unsqueeze(0).to(device)
        feats = feat_extractor(img_batch).cpu().numpy()
        fake_feats.append(feats)
    fake_feats = np.concatenate(fake_feats, axis=0)
    mu_fake = np.mean(fake_feats, axis=0)
    sigma_fake = np.cov(fake_feats, rowvar=False)
    diff = mu_real - mu_fake
    covmean = np.sqrt(sigma_real @ sigma_fake)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    fid = diff @ diff + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

#### Блок GAN:

In [7]:
class GANGenerator(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0, bias=False), 
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Upsample(scale_factor=2), 
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Upsample(scale_factor=2), 
            nn.Conv2d(128, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Upsample(size=(50, 200)), 
            nn.Conv2d(64, 3, 3, padding=1),
            nn.Tanh()
        )
    def forward(self, z):
        return self.main(z.view(z.size(0), self.latent_dim, 1, 1))
class GANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1, bias=False), 
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1, bias=False), 
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, 4, 2, 1, bias=False), 
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(512, 1, 1),
            nn.Flatten()
        )
    def forward(self, img):
        return self.main(img)
def train_gan():
    if weights_exist("gan"):
        print("Веса GAN уже существуют.")
        return
    dataset = CaptchaDataset(TRAIN_DIR, return_label=False)
    print(f"Размера набора данных: {len(dataset)}")
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
    generator = GANGenerator(LATENT_DIM).to(DEVICE)
    discriminator = GANDiscriminator().to(DEVICE)
    g_optim = optim.Adam(generator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
    d_optim = optim.Adam(discriminator.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
    criterion = nn.BCEWithLogitsLoss()
    monitor = ResourceMonitor()
    monitor.start()
    start_time = time.time()
    g_losses = []
    d_losses = []
    print("Обучение GAN:")
    for epoch in range(1, NUM_EPOCHS_GAN+1):
        for i, real_imgs in enumerate(loader):
            real_imgs = real_imgs.to(DEVICE)
            bs = real_imgs.size(0)
            real_labels = torch.full((bs, 1), 1.0, device=DEVICE)
            fake_labels = torch.full((bs, 1), 0.0, device=DEVICE)
            d_optim.zero_grad()
            real_out = discriminator(real_imgs) #Дискриминатор
            d_real_loss = criterion(real_out, real_labels)
            z = torch.randn(bs, LATENT_DIM, device=DEVICE)
            fake_imgs = generator(z).detach()
            fake_out = discriminator(fake_imgs)
            d_fake_loss = criterion(fake_out, fake_labels)
            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_optim.step()
            g_optim.zero_grad()
            z = torch.randn(bs, LATENT_DIM, device=DEVICE) #Генератор
            gen_imgs = generator(z)
            gen_out = discriminator(gen_imgs)
            g_loss = criterion(gen_out, real_labels)
            g_loss.backward()
            g_optim.step()
            d_losses.append(d_loss.item())
            g_losses.append(g_loss.item())
            print(f"Batch {i+1}/{len(loader)}", end='\r')
        if epoch % 5 == 0:
            print(f"Epoch [{epoch}/{NUM_EPOCHS_GAN}] D loss: {d_loss.item():.4f}, G loss: {g_loss.item():.4f}")
    monitor.stop()
    elapsed = time.time() - start_time
    print(f"GAN обучен за {elapsed:.1f} сек.")
    print(monitor.summary())
    if torch.cuda.is_available():
        print(f"Max GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB")
    save_weights(generator, "gan")
    save_loss_plot(g_losses, "GAN Generator Loss", "gan_g_loss.png")
    save_loss_plot(d_losses, "GAN Discriminator Loss", "gan_d_loss.png")
def generate_gan(n_samples=10):
    generator = GANGenerator(LATENT_DIM).to(DEVICE)
    load_weights(generator, "gan")
    generator.eval()
    create_dir(GEN_DIRS["gan"])
    with torch.no_grad():
        for i in range(n_samples):
            z = torch.randn(1, LATENT_DIM, device=DEVICE)
            img = generator(z).cpu()
            img = (img * 0.5) + 0.5 # денормализация
            save_image(img, os.path.join(GEN_DIRS["gan"], f"gan_{i+1}.png"))
    print(f"Сгенерировано {n_samples} изображений GAN в {GEN_DIRS['gan']}")

#### Блок VAE:

In [8]:
class VAEEncoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(9216, latent_dim) #после свёрток размер: для 50x200 -> примерно 256 * 3 * 12 = 9216
        self.fc_logvar = nn.Linear(9216, latent_dim)
    def forward(self, x):
        h = self.conv(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
class VAEDecoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 13)  # 4x13 примерно
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), #64x208 -> нужно 50x200
            nn.Upsample(size=(50, 200), mode='bilinear', align_corners=False),
            nn.Tanh()
        )
    def forward(self, z):
        h = self.fc(z).view(-1, 256, 4, 13)
        return self.deconv(h)
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)
        self.latent_dim = latent_dim
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar
def loss_vae(recon, x, mu, logvar, beta=1.0):
    bce = F.binary_cross_entropy_with_logits(recon, x, reduction='sum') / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return bce + beta * kl
def train_vae():
    if weights_exist("vae"):
        print("Веса VAE уже существуют.")
        return
    dataset = CaptchaDataset(TRAIN_DIR, return_label=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    model = VAE(LATENT_DIM).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    monitor = ResourceMonitor()
    monitor.start()
    start_time = time.time()
    losses = []
    print("Обучение VAE:")
    model.train()
    for epoch in range(1, NUM_EPOCHS_VAE+1):
        for imgs in loader:
            imgs = imgs.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(imgs)
            loss = loss_vae(recon, imgs, mu, logvar)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if epoch % 5 == 0:
            print(f"Epoch [{epoch}/{NUM_EPOCHS_VAE}] loss: {loss.item():.4f}")
    monitor.stop()
    elapsed = time.time() - start_time
    print(f"VAE обучен за {elapsed:.1f} сек.")
    print(monitor.summary())
    if torch.cuda.is_available():
        print(f"Max GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB")
    save_weights(model, "vae")
    save_loss_plot(losses, "VAE Training Loss", "vae_loss.png")
def generate_vae(n_samples=10):
    model = VAE(LATENT_DIM).to(DEVICE)
    load_weights(model, "vae")
    model.eval()
    create_dir(GEN_DIRS["vae"])
    with torch.no_grad():
        for i in range(n_samples):
            z = torch.randn(1, LATENT_DIM, device=DEVICE)
            img = model.decoder(z).cpu()
            img = (img * 0.5) + 0.5
            save_image(img, os.path.join(GEN_DIRS["vae"], f"vae_{i+1}.png"))
    print(f"Сгенерировано {n_samples} изображений VAE в {GEN_DIRS['vae']}")

#### Блок DPM:

In [9]:
def extract(a, t, x_shape):
    batch_size = t.shape[0]
    out = a.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))
class DDPM(nn.Module):
    def __init__(self, img_size=(50,200), channels=3, timesteps=1000): #Упрощённая DDPM с U-Net
        super().__init__()
        self.timesteps = timesteps
        self.img_size = img_size
        beta = torch.linspace(1e-4, 0.02, timesteps)
        alpha = 1. - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('alpha_bar', alpha_bar)
        self.denoise = UNet(channels, channels)
    def forward(self, x, t):
        return self.denoise(x, t)
    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1. - self.alpha_bar)
        return extract(sqrt_alpha_bar, t, x0.shape) * x0 + extract(sqrt_one_minus_alpha_bar, t, x0.shape) * noise, noise
    def p_sample(self, model, xt, t):
        alpha = extract(self.alpha, t, xt.shape)
        alpha_bar = extract(self.alpha_bar, t, xt.shape)
        beta = extract(self.beta, t, xt.shape)
        eps = model(xt, t)
        coeff = 1 / torch.sqrt(alpha)
        coeff_eps = (1 - alpha) / torch.sqrt(1 - alpha_bar)
        mean = coeff * (xt - coeff_eps * eps)
        if t[0] > 0:
            noise = torch.randn_like(xt)
            sigma = torch.sqrt(beta)
            return mean + sigma * noise
        else:
            return mean
    @torch.no_grad()
    def generate(self, model, num_samples, device):
        model.eval()
        xt = torch.randn(num_samples, 3, *self.img_size, device=device)
        for t in reversed(range(self.timesteps)):
            t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
            xt = self.p_sample(model, xt, t_batch)
        return xt
class UNet(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.inc = nn.Conv2d(in_ch, 64, 3, padding=1)
        self.down1 = nn.Conv2d(64, 128, 4, 2, 1)
        self.down2 = nn.Conv2d(128, 256, 4, 2, 1)
        self.bottleneck = nn.Conv2d(256, 256, 3, padding=1)
        self.up1 = nn.ConvTranspose2d(512, 128, 4, 2, 1)  # concat после down2
        self.up2 = nn.ConvTranspose2d(256, 64, 4, 2, 1)   # concat после down1
        self.outc = nn.Conv2d(128, out_ch, 3, padding=1)
        self.time_emb = nn.Sequential(
            nn.Linear(1, 256),
            nn.ReLU(),
            nn.Linear(256, 256)
        )
    def forward(self, x, t):
        te = self.time_emb(t.float().unsqueeze(-1) / 1000.0).unsqueeze(-1).unsqueeze(-1)
        x1 = F.relu(self.inc(x))
        x1 = x1 + te[:, :x1.size(1)]
        d1 = F.relu(self.down1(x1))
        d2 = F.relu(self.down2(d1))
        b  = F.relu(self.bottleneck(d2))
        u1 = self.up1(torch.cat([b, d2], dim=1))
        u1 = F.interpolate(u1, size=d1.shape[2:], mode='bilinear', align_corners=False)
        u2 = F.relu(self.up2(torch.cat([u1, d1], dim=1))) 
        out = self.outc(torch.cat([u2, x1], dim=1)) 
        return out
def train_ddpm():
    if weights_exist("ddpm"):
        print("Веса DDPM уже существуют.")
        return
    dataset = CaptchaDataset(TRAIN_DIR, return_label=False)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    ddpm = DDPM(img_size=(IMG_SIZE[1], IMG_SIZE[0]), channels=3).to(DEVICE)
    model = ddpm.denoise
    optimizer = optim.Adam(model.parameters(), lr=2e-4)
    monitor = ResourceMonitor()
    monitor.start()
    start_time = time.time()
    losses = []
    print("Обучение DDPM:")
    model.train()
    for epoch in range(1, NUM_EPOCHS_DDPM+1):
        for imgs in loader:
            imgs = imgs.to(DEVICE)
            bs = imgs.size(0)
            t = torch.randint(0, ddpm.timesteps, (bs,), device=DEVICE)
            noise = torch.randn_like(imgs)
            xt, _ = ddpm.q_sample(imgs, t, noise)
            pred_noise = model(xt, t)
            loss = F.mse_loss(pred_noise, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        if epoch % 5 == 0:
            print(f"Epoch [{epoch}/{NUM_EPOCHS_DDPM}] loss: {loss.item():.6f}")
    monitor.stop()
    elapsed = time.time() - start_time
    print(f"DDPM обучен за {elapsed:.1f} сек.")
    print(monitor.summary())
    if torch.cuda.is_available():
        print(f"Max GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB")
    save_weights(ddpm, "ddpm")   # сохраняем весь объект, т.к. содержит буферы
    save_loss_plot(losses, "DDPM Training Loss", "ddpm_loss.png")
def generate_ddpm(n_samples=10):
    ddpm = DDPM(img_size=(IMG_SIZE[1], IMG_SIZE[0]), channels=3).to(DEVICE)
    load_weights(ddpm, "ddpm")  # нужна функция load для всего класса
    create_dir(GEN_DIRS["ddpm"])
    samples = ddpm.generate(ddpm.denoise, n_samples, DEVICE)
    for i, img in enumerate(samples):
        img = (img * 0.5 + 0.5).cpu()
        save_image(img, os.path.join(GEN_DIRS["ddpm"], f"ddpm_{i+1}.png"))
    print(f"Сгенерировано {n_samples} изображений DDPM в {GEN_DIRS['ddpm']}")

#### Для более объективной оценки работы моделей имеет смысл обучить дискриминативную модель распознавания текста, чтобы она могла оценить качество сгенерированного каждой из моделей текста.

In [10]:
class CRNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=1, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2), 
            nn.Conv2d(64, 128, 3, stride=1, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(128, 256, 3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2,1)), 
            nn.Conv2d(256, 256, 3, stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2,2)),
        )
        self.lstm = nn.LSTM(256*3, 128, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(256, num_classes)
    def forward(self, x):
        feat = self.cnn(x) #x: [B, C, H, W]
        feat = feat.permute(0, 3, 1, 2)     
        B, W, C, H = feat.size()
        feat = feat.reshape(B, W, C*H) 
        lstm_out, _ = self.lstm(feat)
        logits = self.fc(lstm_out) 
        return logits
def decode_ctc(logits):
    logits = logits.log_softmax(2)
    _, max_indices = torch.max(logits, dim=2)
    raw_preds = []
    for seq in max_indices:
        pred = []
        prev = -1
        for idx in seq:
            if idx != 0 and idx != prev: 
                pred.append(idx.item()-1) 
            prev = idx
        raw_preds.append(pred)
    return raw_preds
def label_to_string(indices):
    return ''.join([IDX_TO_CHAR[i] for i in indices if i < len(CHARS)])
def train_crnn():
    if weights_exist("ocr"):
        print("Веса OCR уже существуют.")
        return
    train_ds = CaptchaDataset(TRAIN_DIR, return_label=True)
    test_ds = CaptchaDataset(TEST_DIR, return_label=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = CRNN(N_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    ctc_loss = nn.CTCLoss(blank=0, reduction='mean')
    monitor = ResourceMonitor()
    monitor.start()
    start_time = time.time()
    train_losses = []
    print("Обучение CRNN:")
    model.train()
    for epoch in range(1, NUM_EPOCHS_CRNN+1):
        epoch_loss = 0
        for imgs, labels in train_loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(imgs)                     # [B, W', N_CLASSES]
            log_probs = logits.log_softmax(2).permute(1,0,2)  # [W', B, N_CLASSES]
            input_lengths = torch.full((logits.size(0),), logits.size(1), dtype=torch.long)
            target_lengths = torch.full((labels.size(0),), CAPTCHA_LEN, dtype=torch.long)
            loss = ctc_loss(log_probs, labels, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
            epoch_loss += loss.item()
        if epoch % 5 == 0:
            print(f"Epoch [{epoch}/{NUM_EPOCHS_CRNN}] loss: {epoch_loss/len(train_loader):.4f}")
    monitor.stop()
    elapsed = time.time() - start_time
    print(f"CRNN обучен за {elapsed:.1f} сек.")
    print(monitor.summary())
    if torch.cuda.is_available():
        print(f"Max GPU memory allocated: {torch.cuda.max_memory_allocated()/1024**2:.0f} MB")
    save_weights(model, "ocr")
    save_loss_plot(train_losses, "CRNN CTC Loss", "ocr_loss.png")
    model.eval() #Оценка на тесте
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs)
            preds = decode_ctc(logits)
            for pred, true in zip(preds, labels.cpu().numpy()):
                if len(pred) != CAPTCHA_LEN:
                    continue
                if all(p == t for p, t in zip(pred, true)):
                    correct += 1
                total += 1
    acc = correct / total if total > 0 else 0
    print(f"Точность распознавания на тесте: {acc:.4f}")
    return model

In [11]:
def evaluate_generative(model_name, gen_folder, recognition_model):
    print(f"\nОценка модели {model_name}...")
    from glob import glob
    files = sorted(glob(os.path.join(gen_folder, "*.png")))
    if len(files) == 0:
        print(f"Нет изображений в {gen_folder}")
        return
    transform = T.Compose([
        T.Resize((IMG_SIZE[1], IMG_SIZE[0])),
        T.ToTensor(),
        T.Normalize((0.5,), (0.5,))
    ])
    images = []
    for f in files:
        img = Image.open(f).convert("RGB")
        img_t = transform(img).unsqueeze(0).to(DEVICE)
        images.append(img_t)
    recognition_model.eval() #Распознаваемость
    probs = []
    decoded = []
    with torch.no_grad():
        for img_t in images:
            logits = recognition_model(img_t)
            preds = decode_ctc(logits)
            decoded.append(preds[0] if len(preds)>0 else [])
            # средняя вероятность символов (после softmax)
            log_probs = F.log_softmax(logits, dim=2)
            best_prob = torch.max(log_probs, dim=2)[0]
            # средняя по последовательности
            seq_prob = torch.mean(best_prob).exp().item()
            probs.append(seq_prob)
    print(f"Средняя уверенность распознавания: {np.mean(probs):.4f}")
    for i, (d, p) in enumerate(zip(decoded, probs)):
        text = label_to_string(d) if d else "??? ???"
        print(f"  Изображение {i+1}: '{text}' (prob {p:.3f})")
    try: #Схожесть (FID) – вычисляем на относительно небольшом наборе тестовых данных
        test_ds = CaptchaDataset(TEST_DIR, return_label=True)
        real_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=0)
        feat_extractor = InceptionFeatureExtractor().to(DEVICE)
        fake_tensors = [img.squeeze(0) for img in images] #подготавка fake_images как список тензоров
        fid = calculate_fid(real_loader, fake_tensors, feat_extractor, DEVICE)
        print(f"FID (мера схожести с реальными): {fid:.2f}")
    except Exception as e:
        print(f"Не удалось вычислить FID: {e}")

In [ ]:
#if __name__ == "__main__":
    #current_model = input('Какую операцию обучения или использования применить: "gan", "vae", "ddpm", "ocr", "generate", "evaluate"?')
train_gan()
generate_gan(10)

Какую операцию обучения или использования применить: "gan", "vae", "ddpm", "ocr", "generate", "evaluate"? gan


Размера набора данных: 8501
Обучение GAN:
Epoch [5/50] D loss: 0.0227, G loss: 4.5946
Epoch [10/50] D loss: 0.0162, G loss: 4.8109
Epoch [15/50] D loss: 0.0033, G loss: 6.8322
Epoch [20/50] D loss: 0.0010, G loss: 7.7274
Epoch [25/50] D loss: 0.0027, G loss: 6.8923
Epoch [30/50] D loss: 0.0005, G loss: 7.8829
Epoch [35/50] D loss: 0.0003, G loss: 8.8148
Epoch [40/50] D loss: 0.0002, G loss: 9.3433
Epoch [45/50] D loss: 0.0001, G loss: 10.1831
Epoch [50/50] D loss: 0.0010, G loss: 7.0901
GAN обучен за 705.9 сек.
CPU usage max: 39.8%, avg: 13.4%
RAM usage max: 60.6%, avg: 55.9%

Max GPU memory allocated: 401 MB
Веса gan сохранены в weights\gan_weights.pth


In [ ]:
train_vae()
generate_vae(10)

Обучение VAE:
Epoch [5/50] loss: 12905.1387
Epoch [10/50] loss: 12743.1885
Epoch [15/50] loss: 12859.1641
Epoch [20/50] loss: 12618.1221
Epoch [25/50] loss: 12024.3584
Epoch [30/50] loss: 12635.6875
Epoch [35/50] loss: 12486.4326
Epoch [40/50] loss: 12409.8926
Epoch [45/50] loss: 12177.4883
Epoch [50/50] loss: 11587.7578
VAE обучен за 623.4 сек.
CPU usage max: 30.6%, avg: 11.3%
RAM usage max: 61.6%, avg: 55.8%

Max GPU memory allocated: 189 MB
Веса vae сохранены в weights\vae_weights.pth


In [ ]:
train_ddpm()
generate_ddpm(10)

Обучение DDPM:
Epoch [5/30] loss: 0.046292
Epoch [10/30] loss: 0.032584
Epoch [15/30] loss: 0.033335
Epoch [20/30] loss: 0.026587
Epoch [25/30] loss: 0.026822
Epoch [30/30] loss: 0.037125
DDPM обучен за 568.1 сек.
CPU usage max: 39.0%, avg: 13.5%
RAM usage max: 57.1%, avg: 56.3%

Max GPU memory allocated: 1260 MB
Веса ddpm сохранены в weights\ddpm_weights.pth


In [ ]:
train_crnn()

Обучение CRNN:
Epoch [5/30] loss: -0.0938
Epoch [10/30] loss: -0.1288
Epoch [15/30] loss: -0.1347
Epoch [20/30] loss: -0.1358
Epoch [25/30] loss: -0.1406
Epoch [30/30] loss: -0.1390
CRNN обучен за 405.1 сек.
CPU usage max: 33.0%, avg: 14.2%
RAM usage max: 62.2%, avg: 60.5%

Max GPU memory allocated: 514 MB
Веса ocr сохранены в weights\ocr_weights.pth


In [12]:
print("Генерация изображений...") #Только генерация (веса должны уже существовать)
#generate_gan(10)
#generate_vae(10)
generate_ddpm(10)

Генерация изображений...
Веса ddpm загружены из weights\ddpm_weights.pth
Сгенерировано 10 изображений DDPM в ddpm_gen_captcha


In [ ]:
if not weights_exist("ocr"): #Загрука модели распознавания и оценка всех генераций
    print("Сначала обучите модель распознавания")
    sys.exit(1)
crnn = CRNN(N_CLASSES).to(DEVICE)
load_weights(crnn, "ocr")
if os.path.isdir(GEN_DIRS["gan"]):
    evaluate_generative("GAN", GEN_DIRS["gan"], crnn)
if os.path.isdir(GEN_DIRS["vae"]):
    evaluate_generative("VAE", GEN_DIRS["vae"], crnn)
if os.path.isdir(GEN_DIRS["ddpm"]):
    evaluate_generative("DDPM", GEN_DIRS["ddpm"], crnn)

Веса ocr загружены из weights\ocr_weights.pth

Оценка модели GAN...
Средняя уверенность распознавания: 0.9682
  Изображение 1: 'l' (prob 0.965)
  Изображение 2: '??? ???' (prob 0.996)
  Изображение 3: 'l' (prob 0.964)
  Изображение 4: '??? ???' (prob 0.949)
  Изображение 5: '??? ???' (prob 0.975)
  Изображение 6: '??? ???' (prob 0.940)
  Изображение 7: '??? ???' (prob 0.988)
  Изображение 8: '??? ???' (prob 0.977)
  Изображение 9: 'l' (prob 0.967)
  Изображение 10: 'l' (prob 0.962)
